# tinyfuse accelerators: Triton (GPU), NKI (Trainium simulator), JAX/XLA + StableHLO (MLIR)

Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
The last cell prints a summary to paste back.

In [ ]:
!nvidia-smi -L
!rm -rf tinyfuse && git clone -q https://github.com/serjankaur05/tinyfuse
%cd tinyfuse

## 1. Triton fused dense kernel vs PyTorch (eager, torch.compile, CUDA graphs)

In [ ]:
!python accel/triton_dense.py --batches 256 4096

## 2. JAX/XLA: StableHLO (MLIR) lowering and XLA fusion analysis

In [ ]:
!python accel/xla_jax.py --batch 256

## 3. NKI kernel in the Neuron simulator
Runs in an isolated Python 3.12 venv (current neuronx-cc wheels are cp311-cp314).

In [ ]:
!pip -q install uv
!uv venv -q --allow-existing /content/nki -p 3.12
!uv pip install -q -p /content/nki numpy neuronx-cc --extra-index-url https://pip.repos.neuron.amazonaws.com --index-strategy unsafe-best-match
!/content/nki/bin/python accel/nki_dense.py

## Summary (paste this back)

In [ ]:
import json, os
out = {}
for f in ['gpu_results.json', 'xla_results.json', 'nki_results.json']:
    p = os.path.join('results', f)
    out[f] = json.load(open(p)) if os.path.exists(p) else 'missing: see the error above'
x = out.get('xla_results.json')
if isinstance(x, dict):
    x['xla_fusions'] = {k: v for k, v in list(x['xla_fusions'].items())[:12]}
print(json.dumps(out, indent=1, default=str))